# RoadScan — resolution audit, no retraining

Compare the two saved YOLO26s checkpoints at 640, 768 and 960 on the fixed development domains. Test images and annotations are never loaded. Choose by worst-domain development mAP50–95, then mean. This is additional development tuning, not new independent evidence. No 92% result is promised.

## 1. Environment

Keep the same Ultralytics release and use the existing Kaggle PyTorch installation. No training API is called.

In [ ]:
import sys, subprocess, json, shutil, gc, hashlib
from pathlib import Path
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.177'])
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import yaml
from PIL import Image
from IPython.display import display
from ultralytics import YOLO
OUT = Path('/kaggle/working/resolution_audit')
OUT.mkdir(exist_ok=True)
assert torch.cuda.is_available(), 'Select a GPU.'
print('GPU:', torch.cuda.get_device_name(0))


## 2. Restore immutable development manifests

Attach notebook 06 output and the original dataset. Rebase only development image paths. Validate that the same files and class order are used. The original training and test splits are unchanged.

In [ ]:
origins = list(Path('/kaggle/input').rglob('phase2_checkpoint.txt'))
assert len(origins) == 1, 'Attach notebook 06 output exactly once.'
BASE = origins[0].parent
roots = [p.parent.parent for p in Path('/kaggle/input').rglob('train/images')
         if (p.parent.parent / 'train/labels').is_dir()]
assert len(roots) == 1, 'Attach original roadscan-yolo dataset.'
SOURCE = roots[0]
CLASSES = ['D00', 'D10', 'D20', 'D40']
CHECKPOINTS = {name: BASE / 'runs' / name / 'weights/best.pt'
               for name in ['yolo26s_768', 'yolo26s_960_finetune']}
assert all(p.is_file() for p in CHECKPOINTS.values())
YAMLS = {}
for domain in ['in_domain_dev', 'china_dev', 'development']:
    table = pd.read_csv(BASE / (domain + '_manifest.csv'))
    paths = [str(SOURCE / str(p).split('/RoadScan_yolo/', 1)[1]) for p in table.path]
    assert len(paths) == len(set(paths)) and all(Path(p).is_file() for p in paths)
    listing = OUT / (domain + '.txt')
    listing.write_text('\n'.join(paths))
    config = OUT / (domain + '.yaml')
    config.write_text(yaml.safe_dump(dict(path=str(SOURCE), train=str(listing),
                                        val=str(listing), names=dict(enumerate(CLASSES)))))
    YAMLS[domain] = str(config)
    print(domain, len(paths))


## 3. Inspect training behaviour

Read both histories without training. Falling training loss with rising validation loss and flat or falling AP supports overfitting. A training probe is diagnostic and must never be reported as independent performance.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for row, name in enumerate(CHECKPOINTS):
    history = pd.read_csv(BASE / 'runs' / name / 'results.csv')
    history.columns = history.columns.str.strip()
    for col, component in enumerate(['box', 'cls']):
        axes[row, col].plot(history.epoch, history[f'train/{component}_loss'], label='train')
        axes[row, col].plot(history.epoch, history[f'val/{component}_loss'], label='development')
        axes[row, col].set_title(name + ': ' + component)
        axes[row, col].legend()
    axes[row, 2].plot(history.epoch, history['metrics/mAP50-95(B)'])
    axes[row, 2].set_title(name + ': development AP')
    display(history.tail(5))
fig.tight_layout()
fig.savefig(OUT / 'training_curves.png', dpi=150)
plt.show()
print((BASE / 'training_gap_diagnostic.json').read_text())


## 4. Evaluate the six configurations

Evaluate both saved checkpoints at three predeclared sizes. Square padding, batch size, confidence floor, NMS IoU and maximum detections stay fixed. This isolates evaluation resolution and avoids different rectangular batch shapes across domains. Save after each evaluation so completed work survives a later error. All metrics here are development estimates.

In [ ]:
rows, class_rows = [], []
for name, checkpoint in CHECKPOINTS.items():
    for size in [640, 768, 960]:
        for domain in ['in_domain_dev', 'china_dev']:
            detector = YOLO(str(checkpoint))
            metrics = detector.val(
                data=YAMLS[domain], imgsz=size, batch=4, device=0, workers=2,
                rect=False, conf=0.001, iou=0.7, max_det=300, nms=True,
                augment=False, plots=False, project=str(OUT / 'evaluations'),
                name=f'{name}_{size}_{domain}')
            rows.append(dict(candidate=name, size=size, domain=domain,
                             mAP50=float(metrics.box.map50), mAP50_95=float(metrics.box.map),
                             precision=float(metrics.box.mp), recall=float(metrics.box.mr)))
            for index, category in enumerate(metrics.box.ap_class_index):
                p, r, ap50, ap = metrics.box.class_result(index)
                class_rows.append(dict(candidate=name, size=size, domain=domain,
                    class_name=CLASSES[int(category)], precision=float(p), recall=float(r),
                    AP50=float(ap50), AP50_95=float(ap)))
            pd.DataFrame(rows).to_csv(OUT / 'domain_results.csv', index=False)
            pd.DataFrame(class_rows).to_csv(OUT / 'class_results.csv', index=False)
            del detector, metrics
            gc.collect()
            torch.cuda.empty_cache()
results = pd.DataFrame(rows)
ranking = results.groupby(['candidate', 'size']).agg(
    worst_domain_AP=('mAP50_95', 'min'), mean_domain_AP=('mAP50_95', 'mean'),
    worst_domain_AP50=('mAP50', 'min')).sort_values(
        ['worst_domain_AP', 'mean_domain_AP'], ascending=False)
ranking.to_csv(OUT / 'ranking.csv')
WINNER, SIZE = ranking.index[0]
SIZE = int(SIZE)
display(results)
display(ranking)
print('Development-selected configuration:', WINNER, SIZE)


## 5. Calibrate and check visual predictions

Select a confidence threshold from the fixed mixed-development F1 curve. This threshold is for deployment and does not change the AP ranking. Predict single images to bound GPU memory. Visual examples are diagnostic, not a substitute for aggregate metrics.

In [ ]:
detector = YOLO(str(CHECKPOINTS[WINNER]))
metrics = detector.val(data=YAMLS['development'], imgsz=SIZE, batch=4,
    device=0, workers=2, rect=False, conf=0.001, iou=0.7, max_det=300,
    nms=True, plots=True, project=str(OUT / 'calibration'), name='selected')
confidence = None
for title, curve in zip(metrics.curves, metrics.curves_results):
    if 'F1' in title:
        x, y = np.asarray(curve[0]), np.asarray(curve[1])
        confidence = float(x[np.argmax(y.mean(axis=0) if y.ndim == 2 else y)])
        break
assert confidence is not None
paths = (OUT / 'development.txt').read_text().splitlines()
for path in paths[:8]:
    detector.predict(path, imgsz=SIZE, conf=confidence, iou=0.7, max_det=300,
                     nms=True, rect=False, device=0, save=True, verbose=False,
                     project=str(OUT), name='examples', exist_ok=True)


## 6. Package and verify

Save the selected checkpoint and its exact inference settings. Verify CPU inference in a fresh process. Report all configurations and the unmet 92% target honestly. No test score is produced and no additional model training takes place.

In [ ]:
bundle = OUT / 'model_bundle'
bundle.mkdir(exist_ok=True)
shutil.copy2(CHECKPOINTS[WINNER], bundle / 'best.pt')
config = dict(classes=CLASSES, image_size=SIZE, confidence=confidence, iou=0.7,
              max_det=300, nms=True, rect=False, candidate=WINNER,
              selection='Development only; no independent local test yet')
(bundle / 'config.json').write_text(json.dumps(config, indent=2))
(bundle / 'requirements.txt').write_text('ultralytics==8.4.177\n')
code = """import argparse, json
from pathlib import Path
from ultralytics import YOLO
parser = argparse.ArgumentParser()
parser.add_argument('--image', required=True)
args = parser.parse_args()
root = Path(__file__).resolve().parent
c = json.loads((root / 'config.json').read_text())
model = YOLO(str(root / 'best.pt'))
model.predict(args.image, imgsz=c['image_size'], conf=c['confidence'],
              iou=c['iou'], max_det=c['max_det'], nms=c['nms'], rect=c['rect'],
              device='cpu', verbose=False)
print('Fresh-process CPU inference passed')
"""
(bundle / 'predict.py').write_text(code)
check = subprocess.run([sys.executable, str(bundle / 'predict.py'), '--image', paths[0]],
                       capture_output=True, text=True, timeout=180)
(OUT / 'fresh_load.txt').write_text(check.stdout + check.stderr)
assert check.returncode == 0, check.stderr
report = '\n\n'.join([
    '# RoadScan resolution audit', '## Full development comparison', results.to_markdown(index=False),
    '## Ranking', ranking.to_markdown(),
    'Selected: ' + WINNER + ', size=' + str(SIZE),
    '92% mAP50 on both development domains: ' + str(bool(ranking.iloc[0].worst_domain_AP50 >= 0.92)),
    'Training probe performance is not an independent score. Training curves show a train/development gap; do not claim zero overfitting.',
    'Resolution was selected on development data. No new blind-test claim is made. A new annotated local evaluation is still required.',
    'No retraining was performed. Fresh-process CPU inference passed.' ])
(OUT / 'REPORT.md').write_text(report)
(bundle / 'MODEL_CARD.md').write_text(report)
(bundle / 'SHA256.json').write_text(json.dumps({p.name: hashlib.sha256(p.read_bytes()).hexdigest()
    for p in bundle.iterdir() if p.is_file()}, indent=2))
shutil.make_archive('/kaggle/working/roadscan_resolution_model', 'zip', root_dir=bundle)
print(report)
